# Day 25 — SQL for Data Science — Advanced Operations
## DDL, DML, Subqueries, Window Functions

**Objective:** Master advanced SQL concepts including Data Definition Language (DDL), Data Manipulation Language (DML), subqueries, window functions

---

## 1. Environment Setup

In [1]:
import sqlite3
import pandas as pd
from datetime import datetime, timedelta
import random

# Create database for this session
conn = sqlite3.connect('banks.db')
cursor = conn.cursor()

print("✅ SQLite database ready!")
print(f"SQLite version: {sqlite3.sqlite_version}")

✅ SQLite database ready!
SQLite version: 3.50.4


## 2. Data Definition Language (DDL)

### What is DDL?
DDL stands for **Data Definition Language**. It's used to define and manage database structures (tables, indexes, views).

### Key DDL Commands:
- **CREATE**: Create new tables, views, indexes
- **ALTER**: Modify existing table structure
- **DROP**: Remove tables, views, indexes
- **TRUNCATE**: Remove all data from a table
- **RENAME**: Rename objects

In [ ]:
Object	      Stores Data?	                   Purpose
Table	        ✅ Yes	                     Stores actual data
View	   ❌ No (stores a query)	         Presents data from one or more tables
Index	   ❌ No (stores lookup information)  Speeds up data retrieval

In [2]:
# CREATE TABLE with advanced features
cursor.execute('''
CREATE TABLE IF NOT EXISTS accounts (
    account_id INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id INTEGER NOT NULL,
    account_number TEXT UNIQUE NOT NULL,
    account_type TEXT CHECK(account_type IN ('Savings', 'Current', 'Fixed Deposit')),
    balance REAL DEFAULT 0.0,
    interest_rate REAL DEFAULT 0.0,
    created_date DATE DEFAULT CURRENT_DATE,
    status TEXT DEFAULT 'Active' CHECK(status IN ('Active', 'Inactive', 'Closed')),
    FOREIGN KEY (customer_id) REFERENCES customers (customer_id)
)
''')

In [3]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS loans (
    loan_id INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id INTEGER NOT NULL,
    loan_amount REAL NOT NULL,
    interest_rate REAL NOT NULL,
    loan_term_months INTEGER NOT NULL,
    disbursement_date DATE,
    status TEXT DEFAULT 'Active' CHECK(status IN ('Active', 'Paid', 'Default')),
    FOREIGN KEY (customer_id) REFERENCES customers (customer_id)
)
''')

In [3]:
print("✅ Tables created successfully!")
cursor.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")
for table in cursor.fetchall():
    print(f"  📊 {table[0]}")

✅ Tables created successfully!
  📊 accounts
  📊 customers
  📊 customers_between1
  📊 loans
  📊 sqlite_sequence
  📊 trans_30_40
  📊 transactions


In [5]:
# ALTER TABLE: Add and modify columns
print("Adding new columns to customers table...")

# Add email column (Note: SQLite doesn't support IF NOT EXISTS for columns)
try:
    cursor.execute('ALTER TABLE customers ADD COLUMN email TEXT')
    print("✅ Added 'email' column")
except sqlite3.OperationalError:
    print("ℹ️ 'email' column already exists")
finally:
    print("just to check whether finally runs all the time or not")

Adding new columns to customers table...
ℹ️ 'email' column already exists
just to check whether finally runs all the time or not


In [10]:
try:
    cursor.execute('ALTER TABLE customers ADD COLUMN phone TEXT')
    print("✅ Added 'phone' column")
except sqlite3.OperationalError:
    print("ℹ️ 'phone' column already exists")

✅ Added 'phone' column


In [11]:
try:
    cursor.execute('ALTER TABLE customers ADD COLUMN is_active BOOLEAN DEFAULT 1')
    print("✅ Added 'is_active' column")
except sqlite3.OperationalError:
    print("ℹ️ 'is_active' column already exists")

✅ Added 'is_active' column


In [18]:
# CREATE INDEX: Improve query performance
cursor.execute('CREATE INDEX IF NOT EXISTS idx_transactions_customer ON transactions(customer_id)')
cursor.execute('CREATE INDEX IF NOT EXISTS idx_transactions_date ON transactions(transaction_date)')
cursor.execute('CREATE INDEX IF NOT EXISTS idx_transactions_type ON transactions(transaction_type)')
cursor.execute('CREATE INDEX IF NOT EXISTS idx_customers_province ON customers(province)')

In [6]:
# seeing the indexes created so far
pd.read_sql_query("SELECT name, tbl_name, sql FROM sqlite_master WHERE type='index' AND sql IS NOT NULL", conn)

# we can't see the actual data stored in the idx_transactions_customer in sqllite
# however the picture is like this
# Index: idx_loans_customer

# customer_id     →    location of matching row(s)
# -------------------------------------------------
# 105             →    loan 1, loan 3, loan 5
# 150             →    loan 4
# 201             →    loan 2


,name,tbl_name,sql
0,idx_transactions_customer,transactions,CREATE INDEX idx_transactions_customer ON tran...
1,idx_transactions_date,transactions,CREATE INDEX idx_transactions_date ON transact...
2,idx_transactions_type,transactions,CREATE INDEX idx_transactions_type ON transact...
3,idx_customers_province,customers,CREATE INDEX idx_customers_province ON custome...


In [17]:
# practice

cursor.execute('''
CREATE TABLE transactions_customer(
transaction_id INT PRIMARY KEY
INDEX idx_transactions_customer (transaction_date)

)
''')

OperationalError: near "INDEX": syntax error

In [3]:
print("✅ Indexes created for better performance:")
cursor.execute("SELECT name FROM sqlite_master WHERE type='index'")
for idx in cursor.fetchall():
    print(f"  🔍 {idx[0]}")

✅ Indexes created for better performance:
  🔍 sqlite_autoindex_accounts_1
  🔍 idx_transactions_customer
  🔍 idx_transactions_date
  🔍 idx_transactions_type
  🔍 idx_customers_province


In [13]:
# CREATE VIEW: Virtual table based on query
# view is the result set of a stored query
# read-only vs. updatable views
# materalized view

# Advantages of view

# to restrict data access
# to make complex queries easy
# to provide data independence 
# to present different views of the same data

cursor.execute('''
CREATE VIEW IF NOT EXISTS vw_customer_summary AS
SELECT 
    c.customer_id,
    c.full_name,
    c.province,
    c.district,
    c.account_type,
    COUNT(t.transaction_id) as transaction_count,
    SUM(t.amount) as total_transactions,
    AVG(t.amount) as avg_transaction,
    MAX(t.amount) as largest_transaction,
    MIN(t.amount) as smallest_transaction
FROM customers c
LEFT JOIN transactions t ON c.customer_id = t.customer_id
GROUP BY c.customer_id
''')

print("✅ View 'vw_customer_summary' created")
cursor.execute("SELECT name FROM sqlite_master WHERE type='view'")
for view in cursor.fetchall():
    print(f"  👁️ {view[0]}")

✅ View 'vw_customer_summary' created
  👁️ vw_customer_summary


In [7]:
df_accounts = pd.read_sql_query("SELECT * FROM vw_customer_summary", conn)
print(df_accounts)

    customer_id       full_name       province    district account_type  \
0             1   Rajesh Sharma        Bagmati   Kathmandu      Savings   
1             2      Sita Thapa        Gandaki     Pokhara      Current   
2             3  Krishna Gurung        Lumbini      Butwal      Savings   
3             4      Sunita Rai          Koshi  Biratnagar      Savings   
4             5   Hari Shrestha        Bagmati    Lalitpur      Current   
5             6   Gita Adhikari  Sudurpashchim   Dhangadhi      Savings   
6             7      Ram Poudel        Karnali     Surkhet      Savings   
7             8      Maya Karki        Bagmati   Bhaktapur      Current   
8             9     Bikram Shah        Gandaki    Lekhnath      Savings   
9            10    Laxmi Tamang     Province 1      Dharan      Savings   
10           11      Saru Thapa        Bagmati         Ktm      Current   
11           12    Rajesh Dahal        Bagmati   Kathmandu      Savings   
12           13      Sita

## 3. Data Manipulation Language (DML)

### What is DML?
DML stands for **Data Manipulation Language**. It's used to handle data in database objects.

### Key DML Commands:
- **INSERT**: Add new rows
- **UPDATE**: Modify existing rows
- **DELETE**: Remove rows
- **MERGE/UPSERT**: Insert or update

In [8]:
# INSERT: Adding sample accounts
accounts_data = [
    (1, 'ACC001', 'Savings', 50000, 4.5, '2024-01-15', 'Active'),
    (2, 'ACC002', 'Current', 75000, 0.0, '2024-02-01', 'Active'),
    (3, 'ACC003', 'Savings', 120000, 5.0, '2024-02-20', 'Active'),
    (4, 'ACC004', 'Savings', 35000, 4.5, '2024-03-05', 'Active'),
    (5, 'ACC005', 'Current', 200000, 0.0, '2024-03-15', 'Active'),
    (6, 'ACC006', 'Fixed Deposit', 500000, 8.5, '2024-04-01', 'Active'),
    (7, 'ACC007', 'Savings', 45000, 4.5, '2024-04-10', 'Inactive'),
    (8, 'ACC008', 'Current', 180000, 0.0, '2024-05-01', 'Active'),
    (9, 'ACC009', 'Savings', 65000, 4.5, '2024-05-15', 'Active'),
    (10, 'ACC010', 'Savings', 88000, 5.0, '2024-06-01', 'Active'),
    (11, 'ACC001', 'Savings', 50000, 4.5, '2024-01-15', 'Active')
]

cursor.executemany('''
INSERT OR IGNORE INTO accounts 
(customer_id, account_number, account_type, balance, interest_rate, created_date, status)
VALUES (?, ?, ?, ?, ?, ?, ?)
''', accounts_data)


# "INSERT OR IGNORE" is a valid SQLite syntax. 
#    It's used here because the `account_number` column has a UNIQUE constraint, 
# and the data being inserted contains duplicates. By using this, if a duplicate is found, 
# that particular row is skipped instead of causing an error and stopping the whole process.


print("✅ Accounts data inserted:")
df_accounts = pd.read_sql_query("SELECT * FROM accounts", conn)
print(df_accounts.head(10))

✅ Accounts data inserted:
   account_id  customer_id account_number   account_type   balance  \
0           1            1         ACC001        Savings   50000.0   
1           2            2         ACC002        Current   75000.0   
2           3            3         ACC003        Savings  120000.0   
3           4            4         ACC004        Savings   35000.0   
4           5            5         ACC005        Current  200000.0   
5           6            6         ACC006  Fixed Deposit  500000.0   
6           7            7         ACC007        Savings   45000.0   
7           8            8         ACC008        Current  180000.0   
8           9            9         ACC009        Savings   65000.0   
9          10           10         ACC010        Savings   88000.0   

   interest_rate created_date    status  
0            4.5   2024-01-15    Active  
1            0.0   2024-02-01    Active  
2            5.0   2024-02-20    Active  
3            4.5   2024-03-05    Ac

In [10]:
customers_data =[
    (12, 'Rajesh Dahal', 'Bagmati', 'Kathmandu', 'Savings', '2024-01-15', 'email@gmail.com', '0987654345','1'),
    (13, 'Sita Thapa', 'Gandaki', 'Pokhara', 'Current', '2024-02-01', 'email@gmail.com', '0987654345','1')
]
cursor.executemany('''
INSERT OR IGNORE INTO customers VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
''', customers_data)

conn.commit()

In [11]:
df_before = pd.read_sql_query("SELECT * FROM customers", conn)
print(df_before)

    customer_id       full_name       province    district account_type  \
0             1   Rajesh Sharma        Bagmati   Kathmandu      Savings   
1             2      Sita Thapa        Gandaki     Pokhara      Current   
2             3  Krishna Gurung        Lumbini      Butwal      Savings   
3             4      Sunita Rai          Koshi  Biratnagar      Savings   
4             5   Hari Shrestha        Bagmati    Lalitpur      Current   
5             6   Gita Adhikari  Sudurpashchim   Dhangadhi      Savings   
6             7      Ram Poudel        Karnali     Surkhet      Savings   
7             8      Maya Karki        Bagmati   Bhaktapur      Current   
8             9     Bikram Shah        Gandaki    Lekhnath      Savings   
9            10    Laxmi Tamang     Province 1      Dharan      Savings   
10           11      Saru Thapa        Bagmati         Ktm      Current   
11           12    Rajesh Dahal        Bagmati   Kathmandu      Savings   
12           13      Sita

In [22]:
# INSERT with SELECT (Copy data)
cursor.execute('''
INSERT INTO loans (customer_id, loan_amount, interest_rate, loan_term_months, disbursement_date, status)
SELECT 
    customer_id,
    balance * 1.5,
    12.5,
    36,
    created_date,
    'Active'
FROM accounts
WHERE account_type = 'Savings' AND balance > 50000
LIMIT 3
''')

print("✅ Sample loan data inserted:")
df_loans = pd.read_sql_query("SELECT * FROM loans", conn)
print(df_loans)

✅ Sample loan data inserted:
   loan_id  customer_id  loan_amount  interest_rate  loan_term_months  \
0       13            3     180000.0           12.5                36   
1       14            9      97500.0           12.5                36   
2       15           10     132000.0           12.5                36   

  disbursement_date  status  
0        2024-02-20  Active  
1        2024-05-15  Active  
2        2024-06-01  Active  


In [18]:
# UPDATE: Modify existing records
print("Before Update - Customer 1:")
df_before = pd.read_sql_query("SELECT * FROM customers WHERE customer_id = 1", conn)
print(df_before)

# Update customer details
cursor.execute('''
UPDATE customers 
SET email = 'rajesh.sharma@email.com', 
    phone = '+977-9841234567'
WHERE customer_id = 1
''')

print("After Update - Customer 1:")
df_before = pd.read_sql_query("SELECT * FROM customers WHERE customer_id = 1", conn)
print(df_before)

Before Update - Customer 1:
   customer_id      full_name province   district account_type created_date  \
0            1  Rajesh Sharma  Bagmati  Kathmandu      Savings   2024-01-15   

  email phone  is_active  
0  None  None          1  
After Update - Customer 1:
   customer_id      full_name province   district account_type created_date  \
0            1  Rajesh Sharma  Bagmati  Kathmandu      Savings   2024-01-15   

                     email            phone  is_active  
0  rajesh.sharma@email.com  +977-9841234567          1  


In [23]:
# Update multiple records
cursor.execute('''
UPDATE accounts 
SET balance = balance * 1.05  -- 5% interest bonus
WHERE account_type = 'Savings' AND status = 'Active'
''')

print("\nAfter Update - Customer 1:")
df_after = pd.read_sql_query("SELECT * FROM customers WHERE customer_id = 1", conn)
print(df_after)

print("\nAll accounts after interest update:")
df_accounts_updated = pd.read_sql_query("SELECT customer_id, account_type, balance FROM accounts", conn)
print(df_accounts_updated)


After Update - Customer 1:
   customer_id      full_name province   district account_type created_date  \
0            1  Rajesh Sharma  Bagmati  Kathmandu      Savings   2024-01-15   

                     email            phone  is_active  
0  rajesh.sharma@email.com  +977-9841234567          1  

All accounts after interest update:
   customer_id   account_type   balance
0            1        Savings   52500.0
1            2        Current   75000.0
2            3        Savings  126000.0
3            4        Savings   36750.0
4            5        Current  200000.0
5            6  Fixed Deposit  500000.0
6            7        Savings   45000.0
7            8        Current  180000.0
8            9        Savings   68250.0
9           10        Savings   92400.0


In [24]:
# DELETE: Remove records
print("Before DELETE - Total customers:", len(pd.read_sql_query("SELECT * FROM customers", conn)))

# Delete inactive customers (soft delete approach - update instead)
cursor.execute('''
DELETE FROM customers 
WHERE customer_id NOT IN (SELECT DISTINCT customer_id FROM transactions)
''')

print("After DELETE - Total customers:", len(pd.read_sql_query("SELECT * FROM customers", conn)))

conn.commit()

Before DELETE - Total customers: 13
After DELETE - Total customers: 10


## 4. Subqueries

### Types of Subqueries:
1. **Scalar Subquery**: Returns single value
2. **Row Subquery**: Returns single row
3. **Table Subquery**: Returns table (used in FROM clause)
4. **Correlated Subquery**: References outer query

In [25]:
# 1. Scalar Subquery: Customers above average transaction
print("Customers with transactions above average:")
df_scalar = pd.read_sql_query('''
SELECT 
    c.full_name,
    SUM(t.amount) as total_spent
FROM customers c
JOIN transactions t ON c.customer_id = t.customer_id
GROUP BY c.customer_id
HAVING SUM(t.amount) > (
    SELECT AVG(amount) * 4  --average transaction amount multiplied by 4
    FROM transactions
)
ORDER BY total_spent DESC
''', conn)
print(df_scalar)

Customers with transactions above average:
        full_name  total_spent
0  Krishna Gurung     165000.0
1   Hari Shrestha     150000.0


In [28]:
# 2. Table Subquery: Transaction summary with ranking
print("Monthly transaction summary:")
df_table = pd.read_sql_query('''
SELECT 
    month,
    transaction_count,
    total_amount,
    ROUND(total_amount / transaction_count, 2) as avg_per_transaction
FROM (
    SELECT 
        strftime('%Y-%m', transaction_date) as month,
        COUNT(*) as transaction_count,
        SUM(amount) as total_amount
    FROM transactions
    GROUP BY month
) monthly_stats
ORDER BY month
''', conn)
print(df_table)

Monthly transaction summary:
     month  transaction_count  total_amount  avg_per_transaction
0  2024-05                 11      402000.0             36545.45
1  2024-06                 14      433000.0             30928.57


In [26]:
# 3. Correlated Subquery: Customers exceeding their own average
print("Transactions exceeding customer's average:")
df_correlated = pd.read_sql_query('''
SELECT 
    t.transaction_id,
    c.full_name,
    t.amount,
    ROUND((
        SELECT AVG(amount) 
        FROM transactions t2 
        WHERE t2.customer_id = t.customer_id
    ), 2) as customer_avg,
    ROUND(t.amount - (
        SELECT AVG(amount) 
        FROM transactions t2 
        WHERE t2.customer_id = t.customer_id
    ), 2) as difference
FROM transactions t
JOIN customers c ON t.customer_id = c.customer_id
WHERE t.amount > (
    SELECT AVG(amount) 
    FROM transactions t2 
    WHERE t2.customer_id = t.customer_id
)
ORDER BY difference DESC
LIMIT 5
''', conn)
print(df_correlated)

Transactions exceeding customer's average:
   transaction_id       full_name    amount  customer_avg  difference
0               6   Hari Shrestha  100000.0       50000.0     50000.0
1               1   Rajesh Sharma   50000.0       28750.0     21250.0
2              12    Laxmi Tamang   60000.0       39000.0     21000.0
3               4  Krishna Gurung   75000.0       55000.0     20000.0
4               9      Ram Poudel   45000.0       26500.0     18500.0


In [27]:
# 4. Subquery with EXISTS: Customers with multiple accounts
print("Customers with multiple accounts:")
df_exists = pd.read_sql_query('''
SELECT 
    c.full_name,
    c.province,
    (
        SELECT COUNT(*) 
        FROM accounts a 
        WHERE a.customer_id = c.customer_id
    ) as account_count
FROM customers c
WHERE EXISTS (
    SELECT 1 
    FROM accounts a2 
    WHERE a2.customer_id = c.customer_id 
    GROUP BY a2.customer_id
    HAVING COUNT(*) > 1
)
''', conn)
print(df_exists)

Customers with multiple accounts:
Empty DataFrame
Columns: [full_name, province, account_count]
Index: []


## 5. Window Functions

### Purpose: Perform calculations across rows related to current row

### Key Window Functions:
- **ROW_NUMBER()**: Sequential row number
- **RANK()**: Rank with gaps
- **DENSE_RANK()**: Rank without gaps
- **LAG()/LEAD()**: Access previous/next rows
- **SUM()/AVG() OVER()**: Running totals
- **NTILE()**: Divide rows into buckets

In [1]:
import pandas as pd

In [4]:
# ROW_NUMBER, RANK, DENSE_RANK
print("Transaction ranking within each customer:")
df_rank = pd.read_sql_query('''
SELECT 
    transaction_id,
    customer_id,
    amount,
    ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY amount DESC) as row_num,
    RANK() OVER (PARTITION BY customer_id ORDER BY amount DESC) as rank,
    DENSE_RANK() OVER (PARTITION BY customer_id ORDER BY amount DESC) as dense_rank
FROM transactions
ORDER BY customer_id, amount DESC
LIMIT 15
''', conn)
print(df_rank)

Transaction ranking within each customer:
    transaction_id  customer_id    amount  row_num  rank  dense_rank
0                1            1   50000.0        1     1           1
1               23            1   30000.0        2     2           2
2                3            1   20000.0        3     3           3
3               13            1   15000.0        4     4           4
4               16            2   35000.0        1     1           1
5                2            2   15000.0        2     2           2
6                7            2    5000.0        3     3           3
7                4            3   75000.0        1     1           1
8               24            3   50000.0        2     2           2
9               14            3   40000.0        3     3           3
10              17            4   28000.0        1     1           1
11               5            4   25000.0        2     2           2
12               6            5  100000.0        1     1     

In [5]:
# LAG and LEAD: Compare with previous/next transactions
print("Transaction comparison with previous and next transactions:")
df_lag_lead = pd.read_sql_query('''
SELECT 
    transaction_id,
    customer_id,
    transaction_date,
    amount,
    LAG(amount, 1) OVER (PARTITION BY customer_id ORDER BY transaction_date) as prev_amount,
    LEAD(amount, 1) OVER (PARTITION BY customer_id ORDER BY transaction_date) as next_amount,
    ROUND(amount - LAG(amount, 1) OVER (PARTITION BY customer_id ORDER BY transaction_date), 2) as diff_from_prev
FROM transactions
ORDER BY customer_id, transaction_date
LIMIT 10
''', conn)
print(df_lag_lead)

Transaction comparison with previous and next transactions:
   transaction_id  customer_id transaction_date   amount  prev_amount  \
0               1            1       2024-05-20  50000.0          NaN   
1               3            1       2024-05-22  20000.0      50000.0   
2              13            1       2024-06-02  15000.0      20000.0   
3              23            1       2024-06-12  30000.0      15000.0   
4               2            2       2024-05-21  15000.0          NaN   
5               7            2       2024-05-26   5000.0      15000.0   
6              16            2       2024-06-05  35000.0       5000.0   
7               4            3       2024-05-23  75000.0          NaN   
8              14            3       2024-06-03  40000.0      75000.0   
9              24            3       2024-06-13  50000.0      40000.0   

   next_amount  diff_from_prev  
0      20000.0             NaN  
1      15000.0        -30000.0  
2      30000.0         -5000.0  
3   

In [6]:
# Running totals and moving averages
print("Running totals by customer:")
df_running = pd.read_sql_query('''
SELECT 
    transaction_id,
    customer_id,
    transaction_date,
    amount,
    SUM(amount) OVER (PARTITION BY customer_id ORDER BY transaction_date) as running_total,
    AVG(amount) OVER (PARTITION BY customer_id ORDER BY transaction_date ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) as moving_avg_3,
    COUNT(*) OVER (PARTITION BY customer_id ORDER BY transaction_date) as transaction_number
FROM transactions
WHERE customer_id = 1
ORDER BY transaction_date
''', conn)
print(df_running)

Running totals by customer:
   transaction_id  customer_id transaction_date   amount  running_total  \
0               1            1       2024-05-20  50000.0        50000.0   
1               3            1       2024-05-22  20000.0        70000.0   
2              13            1       2024-06-02  15000.0        85000.0   
3              23            1       2024-06-12  30000.0       115000.0   

   moving_avg_3  transaction_number  
0  50000.000000                   1  
1  35000.000000                   2  
2  28333.333333                   3  
3  21666.666667                   4  


In [ ]:
# NTILE: Divide transactions into quartiles/percentiles
print("Transaction amount quartiles:")
df_ntile = pd.read_sql_query('''
SELECT 
    transaction_id,
    amount,
    NTILE(4) OVER (ORDER BY amount) as quartile,
    NTILE(10) OVER (ORDER BY amount) as percentile,
    CASE 
        WHEN NTILE(4) OVER (ORDER BY amount) = 1 THEN 'Low'
        WHEN NTILE(4) OVER (ORDER BY amount) = 2 THEN 'Medium-Low'
        WHEN NTILE(4) OVER (ORDER BY amount) = 3 THEN 'Medium-High'
        ELSE 'High'
    END as amount_category
FROM transactions
ORDER BY amount
LIMIT 15
''', conn)
print(df_ntile)

## 7. Advanced DDL Operations

### DROP, TRUNCATE, and RENAME

In [ ]:
# RENAME: Rename objects
try:
    cursor.execute('ALTER TABLE accounts RENAME TO accounts_archive')
    print("✅ Renamed 'accounts' to 'accounts_archive'")
    cursor.execute('ALTER TABLE accounts_archive RENAME TO accounts')
    print("✅ Renamed back to 'accounts'")
except sqlite3.OperationalError as e:
    print(f"ℹ️ Rename not needed: {e}")

# DROP: Remove objects
try:
    cursor.execute('DROP TABLE IF EXISTS temp_accounts')
    print("✅ Dropped temporary table if existed")
except sqlite3.OperationalError as e:
    print(f"ℹ️ Nothing to drop: {e}")

In [ ]:
# TRUNCATE alternative in SQLite (DELETE without WHERE)
print("Creating temp table for TRUNCATE demo...")
cursor.execute('''
CREATE TEMP TABLE temp_demo AS
SELECT * FROM accounts WHERE 1=0
''')

cursor.execute("INSERT INTO temp_demo SELECT * FROM accounts LIMIT 3")
print(f"Before TRUNCATE: {len(pd.read_sql_query('SELECT * FROM temp_demo', conn))} rows")

# TRUNCATE alternative
cursor.execute('DELETE FROM temp_demo')
print(f"After TRUNCATE: {len(pd.read_sql_query('SELECT * FROM temp_demo', conn))} rows")

cursor.execute('DROP TABLE temp_demo')
print("✅ Cleaned up temp table")

## 8. Transaction Management

### ACID Properties:
- **Atomicity**: All or nothing
- **Consistency**: Valid state before and after
- **Isolation**: Concurrent transactions don't interfere
- **Durability**: Committed changes persist

In [ ]:
# Transaction with BEGIN, COMMIT, ROLLBACK
print("Transaction Demo: Processing a transfer")

try:
    # Start transaction
    cursor.execute('BEGIN TRANSACTION')
    
    # Get current balances
    cursor.execute("SELECT balance FROM accounts WHERE customer_id = 1")
    balance_from = cursor.fetchone()[0]
    
    cursor.execute("SELECT balance FROM accounts WHERE customer_id = 2")
    balance_to = cursor.fetchone()[0]
    
    transfer_amount = 10000
    
    print(f"  From account balance: {balance_from}")
    print(f"  To account balance: {balance_to}")
    print(f"  Transfer amount: {transfer_amount}")
    
    # Perform transfer
    cursor.execute('''
    UPDATE accounts 
    SET balance = balance - ? 
    WHERE customer_id = 1
    ''', (transfer_amount,))
    
    cursor.execute('''
    UPDATE accounts 
    SET balance = balance + ? 
    WHERE customer_id = 2
    ''', (transfer_amount,))
    
    # Insert transfer record
    cursor.execute('''
    INSERT INTO transactions 
    (customer_id, transaction_date, transaction_type, amount, branch, description)
    VALUES 
    (1, date('now'), 'Transfer', ?, 'Transfer', ?),
    (2, date('now'), 'Deposit', ?, 'Transfer', ?)
    ''', (transfer_amount, 'Transfer to Customer 2', transfer_amount, 'Transfer from Customer 1'))
    
    # Commit transaction
    conn.commit()
    print("✅ Transfer completed successfully!")
    
    # Show updated balances
    cursor.execute("SELECT balance FROM accounts WHERE customer_id = 1")
    new_balance_from = cursor.fetchone()[0]
    cursor.execute("SELECT balance FROM accounts WHERE customer_id = 2")
    new_balance_to = cursor.fetchone()[0]
    
    print(f"  New from balance: {new_balance_from}")
    print(f"  New to balance: {new_balance_to}")
    
except Exception as e:
    # Rollback on error
    conn.rollback()
    print(f"❌ Error occurred: {e}")
    print("Rolled back transaction")

## 9. Performance Optimization Techniques

### 1. Use EXPLAIN to analyze queries
### 2. Create appropriate indexes
### 3. Use CTEs over subqueries when possible
### 4. Limit columns in SELECT

In [ ]:
# EXPLAIN QUERY PLAN - See how SQL executes your query
print("Query Execution Plan:")
cursor.execute('''
EXPLAIN QUERY PLAN
SELECT 
    c.full_name,
    COUNT(t.transaction_id),
    SUM(t.amount)
FROM customers c
JOIN transactions t ON c.customer_id = t.customer_id
WHERE c.province = 'Bagmati'
GROUP BY c.customer_id
HAVING SUM(t.amount) > 100000
ORDER BY SUM(t.amount) DESC
''')

plan = cursor.fetchall()
for row in plan:
    print(f"  {row}")

In [ ]:
# Performance comparison: Subquery vs CTE vs JOIN
print("Performance Comparison (Similar results, different approaches):")

# Approach 1: Subquery
print("\n1. Using Subquery:")
df_sub = pd.read_sql_query('''
SELECT 
    customer_id,
    (SELECT full_name FROM customers c WHERE c.customer_id = t.customer_id) as name,
    SUM(amount) as total
FROM transactions t
GROUP BY customer_id
''', conn)
print(f"   {len(df_sub)} rows returned")

# Approach 2: CTE
print("\n2. Using CTE:")
df_cte_comp = pd.read_sql_query('''
WITH customer_total AS (
    SELECT 
        customer_id,
        SUM(amount) as total
    FROM transactions
    GROUP BY customer_id
)
SELECT 
    c.customer_id,
    c.full_name,
    ct.total
FROM customers c
LEFT JOIN customer_total ct ON c.customer_id = ct.customer_id
''', conn)
print(f"   {len(df_cte_comp)} rows returned")

# Approach 3: Direct JOIN
print("\n3. Using Direct JOIN:")
df_join_comp = pd.read_sql_query('''
SELECT 
    c.customer_id,
    c.full_name,
    SUM(t.amount) as total
FROM customers c
LEFT JOIN transactions t ON c.customer_id = t.customer_id
GROUP BY c.customer_id
''', conn)
print(f"   {len(df_join_comp)} rows returned")

In [ ]:
# Challenge 2: Transaction Pattern Mining
print("\n" + "="*80)
print("📊 Transaction Pattern Mining:")
q2 = '''
WITH transaction_patterns AS (
    SELECT 
        customer_id,
        transaction_date,
        amount,
        transaction_type,
        LAG(transaction_type) OVER (PARTITION BY customer_id ORDER BY transaction_date) as prev_type,
        LAG(amount) OVER (PARTITION BY customer_id ORDER BY transaction_date) as prev_amount,
        LEAD(transaction_type) OVER (PARTITION BY customer_id ORDER BY transaction_date) as next_type,
        ROUND(AVG(amount) OVER (PARTITION BY customer_id ORDER BY transaction_date ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 2) as moving_avg_3,
        SUM(amount) OVER (PARTITION BY customer_id ORDER BY transaction_date) as cumulative_spend
    FROM transactions
),
pattern_stats AS (
    SELECT 
        *,
        CASE 
            WHEN prev_type IS NULL AND next_type IS NOT NULL THEN 'First Transaction'
            WHEN prev_type IS NOT NULL AND next_type IS NULL THEN 'Last Transaction'
            WHEN prev_type = 'Deposit' AND transaction_type = 'Withdrawal' THEN 'Deposit-Withdrawal Pattern'
            WHEN prev_type = 'Withdrawal' AND transaction_type = 'Deposit' THEN 'Withdrawal-Deposit Pattern'
            WHEN amount > moving_avg_3 * 2 THEN 'Unusual Spike'
            WHEN amount < moving_avg_3 * 0.5 THEN 'Unusual Drop'
            ELSE 'Normal Pattern'
        END as pattern_type
    FROM transaction_patterns
)
SELECT 
    pattern_type,
    COUNT(*) as occurrence_count,
    ROUND(AVG(amount), 2) as avg_amount,
    ROUND(AVG(cumulative_spend), 2) as avg_cumulative_spend,
    ROUND(MAX(amount), 2) as max_amount,
    ROUND(MIN(amount), 2) as min_amount
FROM pattern_stats
WHERE pattern_type != 'Normal Pattern'
GROUP BY pattern_type
ORDER BY occurrence_count DESC
'''
try:
    df_q2 = pd.read_sql_query(q2, conn)
    print(df_q2)
except Exception as e:
    print(f"ℹ️ Window function error: {e}")
    print("Using simplified pattern analysis...")
    q2_simple = '''
    SELECT 
        transaction_type,
        COUNT(*) as count,
        ROUND(AVG(amount), 2) as avg_amount,
        ROUND(SUM(amount), 2) as total_amount
    FROM transactions
    GROUP BY transaction_type
    '''
    df_q2 = pd.read_sql_query(q2_simple, conn)
    print(df_q2)

## 10. Summary & Key Takeaways

### 🎓 What We Learned Today:

1. **DDL Operations**: CREATE, ALTER, DROP, RENAME, TRUNCATE
2. **DML Operations**: INSERT, UPDATE, DELETE with advanced features
3. **Subqueries**: Scalar, row, table, and correlated subqueries
4. **Window Functions**: ROW_NUMBER, RANK, LAG, LEAD, running totals
5. **CTEs**: Basic, multiple, and recursive CTEs
6. **Transaction Management**: ACID properties, BEGIN, COMMIT, ROLLBACK
7. **Performance Optimization**: Indexes, EXPLAIN, query optimization

### 📌 Real-World Applications:
- **Banking**: Risk assessment, customer segmentation, fraud detection
- **Finance**: Portfolio management, regulatory compliance
- **Analytics**: Predictive modeling, customer lifetime value

### 🔑 Key SQL Tips for Data Science:
- Use CTEs for complex queries - they're more readable and reusable
- Window functions are powerful for time-series and ranking analysis
- Always use transactions for data modifications
- Create appropriate indexes for frequent queries
- Use EXPLAIN to optimize slow queries

### 📚 Next Steps:
- Practice with real-world datasets
- Implement stored procedures and functions
- Learn about database design and normalization
- Explore NoSQL databases for unstructured data

In [ ]:
# Transaction Management: ACID properties, BEGIN, COMMIT, ROLLBACK
# Performance Optimization: Indexes, EXPLAIN, query optimization
# Files related to these topics are available in the linkedin file so i downloaded it in the folder

In [ ]:
# List all objects in the database
print("\n📚 Database Objects Summary:")
cursor.execute("SELECT name, type FROM sqlite_master WHERE type IN ('table', 'view', 'index') ORDER BY type, name")
for obj in cursor.fetchall():
    emoji = '📊' if obj[1] == 'table' else '👁️' if obj[1] == 'view' else '🔍'
    print(f"  {emoji} {obj[0]} ({obj[1]})")

# Close connection
conn.close()
print("\n✅ Database connection closed")